# ML12 · 進階選修：表徵與生成

從變分自編碼器（VAE）出發，建立解耦表徵、擴散模型與整個生成模型家族的高層直覺，並學會如何連到實務生成式工作流。本書重觀念與直覺，所有資料皆用 numpy 自造，不寫完整訓練。

## 學習目標
- 理解表徵學習（representation learning）的目標：學到「有用、可解釋、可遷移」的潛在特徵。
- 說明 β-VAE 如何透過調整 KL 權重，在重建與解耦（disentanglement）之間取得平衡。
- 用直覺描述擴散模型（diffusion model）的核心：前向加噪與反向去噪。
- 建立生成模型（generative model）全景圖：VAE、GAN、自回歸（autoregressive）、擴散模型各自的取捨。
- 能將上述概念連到實務生成式工作流（generative workflow），知道何時用哪一類模型。

In [ ]:
# 概念：環境設定。固定亂數種子讓結果可重現，並讓 matplotlib 圖內嵌顯示
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(12)   # 固定種子，使每次執行的隨機資料一致，方便對照講解

# 技巧：圖上若要顯示中文標題，需指定有中文字型；此處統一用英文標籤避免缺字方框
plt.rcParams["axes.unicode_minus"] = False   # 避免負號顯示成方框

print("numpy version:", np.__version__)
print("環境就緒，後續資料皆以此 rng 自造")

## 表徵學習的概念與用途

表徵學習（representation learning）是指讓模型自己學出一組「描述資料的座標系」，而不是直接背單一任務的答案。學到的這組座標就是潛在空間（latent space）。

為什麼要學：一組好的表徵能讓下游任務（downstream task）如分類、分群、檢索都變簡單；同一組特徵還能遷移（transfer）到不同任務。特徵抽取（feature extraction）就是把高維原始資料壓成這種低維、好用的座標。

判斷表徵好壞的直覺：相似的資料在潛在空間裡會自然靠在一起。

In [ ]:
# 概念：把高維「建築量體」特徵壓到 2 維潛在座標，觀察相似量體是否自然聚在一起
# 造一批假建築量體：每棟有 樓高(公尺) / 面積(平方公尺) / 開窗比例(0~1) 三個特徵
# 刻意分成兩種典型：高瘦高層、矮胖低層，看它們在潛在空間是否分開
tall_thin = rng.normal(loc=[80, 90, 0.6], scale=[8, 10, 0.05], size=(15, 3))
short_fat = rng.normal(loc=[15, 220, 0.3], scale=[3, 25, 0.05], size=(15, 3))
volumes = np.vstack([tall_thin, short_fat])   # 疊成 30 棟 × 3 特徵的矩陣

# 先標準化（逐欄 z-score），讓不同數量級的特徵公平參與壓縮
z = (volumes - volumes.mean(axis=0)) / volumes.std(axis=0)

# 概念：用 PCA 直覺做最簡降維——取共變異數矩陣特徵向量的前兩個主軸
cov = np.cov(z, rowvar=False)              # rowvar=False：每欄是一個變數（特徵）
eigvals, eigvecs = np.linalg.eigh(cov)     # eigh 適用對稱矩陣，回傳由小到大的特徵值
top2 = eigvecs[:, ::-1][:, :2]             # 反轉取最大兩個方向，即資訊量最大的兩個主軸
latent = z @ top2                          # 投影到 2 維潛在座標

print("原始特徵 shape:", volumes.shape, " -> 潛在座標 shape:", latent.shape)

plt.figure(figsize=(5, 4))
plt.scatter(latent[:15, 0], latent[:15, 1], label="tall-thin")
plt.scatter(latent[15:, 0], latent[15:, 1], label="short-fat")
plt.xlabel("latent dim 1"); plt.ylabel("latent dim 2")
plt.title("Building volumes in 2D latent space"); plt.legend()
plt.tight_layout(); plt.show()

## 從 VAE 到 β-VAE：用 KL 權重換取解耦

變分自編碼器（VAE, variational autoencoder）把資料壓進一個機率化的潛在空間，再從中重建回來。它的損失有兩部分：重建損失（reconstruction loss）要求還原得像，KL 散度（KL divergence）要求潛在分布貼近標準常態。

β-VAE 在 KL 項前加一個權重 β：loss = 重建 + β · KL。β 變大會壓縮潛在資訊、逼每個維度更獨立（更解耦），但重建會變糊。這是一場「解耦（disentanglement）vs 還原」的拉鋸。

公式（僅示意，不訓練）：loss = reconstruction + β · KL。

In [ ]:
# 概念：用自造的兩個生成因子（樓高、胖瘦）資料，示意 β 由小到大時的「解耦 vs 重建」取捨
# 造兩個彼此獨立的真實生成因子：height（樓高）與 chubby（胖瘦）
n = 200
height = rng.uniform(-1, 1, size=n)        # 真實因子 1
chubby = rng.uniform(-1, 1, size=n)        # 真實因子 2（與因子 1 獨立）
factors = np.stack([height, chubby], axis=1)

# 用一個簡化模型示意：β 越大，潛在維度越「對齊單一因子」但重建誤差越高
# alignment 越接近 1 代表某潛在維度幾乎只反映單一生成因子（更解耦）
def simulate_beta(beta):
    align = 0.5 + 0.45 * (1 - np.exp(-beta))        # 隨 β 增大趨近於高度對齊
    recon_err = 0.05 + 0.25 * (1 - np.exp(-beta))   # 注意：β 越大重建越糊（誤差升高）
    return align, recon_err

print("beta  | 因子對齊度 | 重建誤差")
for beta in [0.5, 1.0, 4.0, 8.0]:
    align, err = simulate_beta(beta)
    print(f"{beta:>4.1f}  |   {align:.3f}   |  {err:.3f}")

betas = np.linspace(0.1, 10, 50)
aligns = [simulate_beta(b)[0] for b in betas]
errs = [simulate_beta(b)[1] for b in betas]
plt.figure(figsize=(5, 4))
plt.plot(betas, aligns, label="disentanglement (alignment)")
plt.plot(betas, errs, label="reconstruction error")
plt.xlabel("beta (KL weight)"); plt.ylabel("score")
plt.title("beta-VAE trade-off (illustrative)"); plt.legend()
plt.tight_layout(); plt.show()

## 解讀與評估解耦表徵

解耦的價值在於「轉一個旋鈕只改一件事」：理想上每個潛在維度只控制一個語意因子。要檢查這件事，最直觀的方法是潛在遍歷（latent traversal）。

潛在遍歷：固定其他維度，只掃動單一潛在維度，看輸出怎麼變。若掃動某維度時只有「樓高」單調改變、其他屬性不動，代表這個維度與該因子對齊（factor alignment）良好，可解釋性（interpretability）高。

何時用：在比較不同模型或不同 β 設定時，遍歷圖是判斷解耦好壞最直接的工具。

In [ ]:
# 概念：潛在遍歷——固定其他維度、只掃動單一維度，觀察是否只有目標屬性改變
# 用一個示意解碼器把 2 維潛在座標 (z0, z1) 映回三個可觀察屬性
# 設計成「解耦良好」：z0 幾乎只決定樓高，z1 幾乎只決定胖瘦
def decode(z0, z1):
    height = 10 + 35 * (z0 + 1)        # 樓高主要由 z0 控制
    width = 8 + 4 * (z1 + 1)           # 寬幅（胖瘦）主要由 z1 控制
    window = 0.4 + 0.02 * z1           # 開窗比例只受 z1 微弱影響
    return height, width, window

sweep = np.linspace(-1, 1, 9)          # 掃動範圍：把單一維度從 -1 拉到 1
fixed_z1 = 0.0                         # 注意：遍歷時其他維度必須固定，才能歸因到被掃的維度
heights = [decode(z0, fixed_z1)[0] for z0 in sweep]
widths = [decode(z0, fixed_z1)[1] for z0 in sweep]

print("掃動 z0（固定 z1）時的樓高:", np.round(heights, 1))
print("掃動 z0（固定 z1）時的寬幅:", np.round(widths, 1), "（應幾乎不變）")

plt.figure(figsize=(5, 4))
plt.plot(sweep, heights, marker="o", label="height (should change)")
plt.plot(sweep, widths, marker="s", label="width (should stay flat)")
plt.xlabel("z0 (traversed latent dim)"); plt.ylabel("attribute value")
plt.title("Latent traversal: one knob, one change"); plt.legend()
plt.tight_layout(); plt.show()

## 擴散模型的高層直覺

擴散模型（diffusion model）把生成想成兩個方向相反的過程。前向過程（forward process）一步步把乾淨資料加上高斯雜訊，最後變成幾乎純雜訊；反向過程（reverse process）則學會一步步去噪，把雜訊還原回資料。

關鍵直覺：生成是漸進的去噪步驟（denoising steps），不是一次到位。與 VAE 對照——VAE 用一次編碼/解碼往返，擴散則用很多小步累積。

本節用一維訊號示意加噪與去噪的方向性，不寫真正的訓練。

In [ ]:
# 概念：對一條一維「沿街天際線高度曲線」逐步加噪（前向），再示意逐步去噪（反向）
x = np.linspace(0, 4 * np.pi, 120)
skyline = 30 + 12 * np.sin(x) + 5 * np.sin(2.7 * x)   # 自造一條起伏的天際線高度

# 前向過程：逐步注入高斯雜訊，越後面的步驟越接近純雜訊
steps = 4
noisy_versions = [skyline]
current = skyline.copy()
for t in range(steps):
    current = current + rng.normal(0, 4, size=skyline.shape)   # 每步疊加一點雜訊
    noisy_versions.append(current.copy())

# 反向過程（示意）：用簡單移動平均當「去噪器」，一步步把雜訊版本拉回平滑
def denoise_once(signal, k=7):
    kernel = np.ones(k) / k                       # 注意：這只是示意去噪，真正模型是學出來的
    return np.convolve(signal, kernel, mode="same")

recovered = noisy_versions[-1].copy()
for _ in range(steps):
    recovered = denoise_once(recovered)           # 反覆去噪，逐步逼近原訊號

print("原始與最終去噪結果的均方差:", round(float(np.mean((skyline - recovered) ** 2)), 2))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(skyline, label="clean")
axes[0].plot(noisy_versions[-1], alpha=0.7, label="after forward noising")
axes[0].set_title("Forward: clean -> noise"); axes[0].legend()
axes[1].plot(skyline, label="clean")
axes[1].plot(recovered, alpha=0.8, label="after reverse denoising")
axes[1].set_title("Reverse: noise -> recovered"); axes[1].legend()
plt.tight_layout(); plt.show()

## 生成模型全景與取捨

生成模型（generative model）是一個家族，常見四類：VAE、GAN、自回歸（autoregressive）、擴散模型。沒有萬用模型，差別在取捨。

用四個面向比較選型：樣本品質（quality）、多樣性（diversity）、可控性（controllability）、訓練穩定度（stability）。例如 GAN 品質高但訓練不穩；擴散品質與多樣性都好但生成慢；VAE 穩定可控但樣本偏糊；自回歸品質好但逐元素生成慢。

下一個 cell 用一張自造的對照表整理相對強弱與各自適用情境。

In [ ]:
# 概念：用字典/list 自造一張四類生成模型的四面向比較表，並各舉一個適用情境
# 分數為相對示意（1=弱, 5=強），非實測數據，用來建立選型直覺
models = ["VAE", "GAN", "Autoregressive", "Diffusion"]
scores = {
    "quality":         [3, 5, 4, 5],
    "diversity":       [4, 2, 4, 5],
    "controllability": [4, 3, 3, 4],
    "stability":       [5, 2, 4, 4],
}
use_case = {
    "VAE": "需要平滑可控潛在空間、能做表徵與插值",
    "GAN": "追求最銳利逼真的單張樣本、可接受訓練調參成本",
    "Autoregressive": "序列型資料、逐步生成且要高保真",
    "Diffusion": "要高品質又高多樣性、可接受較慢生成",
}

header = "model           | qual | dive | ctrl | stab"
print(header); print("-" * len(header))
for i, m in enumerate(models):
    # 技巧：用 f-string 對齊欄位，純文字表也能讀得清楚
    print(f"{m:<15} |  {scores['quality'][i]}   |  {scores['diversity'][i]}   |"
          f"  {scores['controllability'][i]}   |  {scores['stability'][i]}")

print("\n各自適用情境:")
for m in models:
    print(f"  - {m}: {use_case[m]}")

## 實務生成式工作流

實務上很少只用單一模型，而是一條生成式工作流（generative workflow）：表徵 + 生成 + 條件控制 + 評估。理解概念才能在工具鏈中做對選擇與把關。

幾個關鍵環節：條件生成（conditional generation）讓你用條件向量指定想要的屬性；潛在空間操控（latent manipulation）讓你在潛在座標上微調語意；最後一定要評估與人工把關（human-in-the-loop），由人對候選結果排序與篩選。

下一個 cell 用一條最小工作流骨架（pseudo-pipeline）示意：從條件向量到生成候選、再到評分排序與人工把關，資料與分數皆自造。

In [ ]:
# 概念：最小生成式工作流骨架——條件向量 -> 生成候選 -> 評分排序 -> 人工把關
# 條件向量：指定想要的目標屬性（此例：偏好高樓、適中開窗比例）
condition = {"target_height": 60.0, "target_window": 0.5}

# 步驟 1：用條件加隨機擾動生成一批候選量體（示意「條件生成」）
def generate_candidates(cond, k=5):
    base = np.array([cond["target_height"], cond["target_window"]])
    noise = rng.normal(0, [10, 0.1], size=(k, 2))   # 在條件附近採樣多個候選
    return base + noise

# 步驟 2：對每個候選評分（示意評估）——越貼近條件分數越高
def score(candidate, cond):
    target = np.array([cond["target_height"], cond["target_window"]])
    dist = np.linalg.norm((candidate - target) / np.array([20, 0.2]))   # 正規化距離
    return float(np.exp(-dist))                                          # 距離越小分數越接近 1

candidates = generate_candidates(condition)
scores = np.array([score(c, condition) for c in candidates])

# 步驟 3：依分數由高到低排序（自動排序，供人工把關）
order = np.argsort(scores)[::-1]
print("候選量體 (height, window) 與分數，已依分數排序:")
for rank, idx in enumerate(order, start=1):
    h, w = candidates[idx]
    print(f"  rank {rank}: height={h:5.1f}, window={w:.2f}, score={scores[idx]:.3f}")

# 步驟 4：人工把關——人通常只從前幾名挑選，這裡示意取 top-2 交付審查
print("\n交付人工審查的 top-2 候選 index:", order[:2].tolist())

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請自己用 numpy 造（仿真即可），完成後對照「驗收」確認輸出。

In [ ]:
# TODO 1 ·（簡單）量體的潛在地圖（整合：表徵學習 + 潛在空間）
#   情境：你有一批自造的都市建築量體（樓高、基地面積、容積率），想看看它們在潛在空間中怎麼分布。
#   要求：
#     1. 用 numpy 自造約 30 筆建築量體向量（可刻意分成「高瘦高層」與「矮胖低層」兩種典型）。
#     2. 先逐欄標準化，再把它們投影到 2 維潛在座標（簡單降維或自訂壓縮皆可），畫出散布圖。
#   驗收：應該看到相似量體在潛在地圖上分成可辨識的群（兩個明顯群落）。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）解耦旋鈕測試（整合：beta-VAE + KL 權重 + 解耦 + 潛在遍歷）
#   情境：你想驗證調大 KL 權重 beta 是否真的讓「樓高」變成一個獨立可控的旋鈕。
#   要求：
#     1. 自造帶有兩個生成因子（樓高、胖瘦）的量體資料。
#     2. 用兩組不同 beta（小、大）的示意設定，比較重建誤差與潛在維度對齊程度（可用數值表）。
#     3. 對較佳的一組做潛在遍歷：固定其他維度、只掃動一個維度，觀察是否只有樓高改變。
#   驗收：應看到大 beta 時重建較糊但某維度幾乎只控制樓高，呈現解耦與重建的取捨。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）為都市量體選一條生成路線（整合：擴散模型直覺 + 生成模型全景 + 生成式工作流 + 表徵）
#   情境：團隊要替「沿街天際線量體」做生成式發想，需在 VAE、GAN、自回歸、擴散模型中選型並設計最小工作流。
#   要求：
#     1. 自造一條一維天際線高度曲線，示意對它前向加噪再反向去噪的方向（畫圖或印均方差皆可）。
#     2. 用四面向（品質、多樣性、可控性、穩定度）比較表為此任務挑選一類生成模型，並寫出選型理由。
#     3. 設計一個「條件向量 -> 生成候選 -> 評分排序 -> 人工把關」的最小工作流骨架（流程與假分數自造）。
#   驗收：應看到一條有理由的選型結論，加上一個可解釋、含評估把關的生成式工作流草圖。
# TODO: 在這裡完成


## 小結

- 表徵學習的目標是學出「資料的內在座標系」（潛在空間），讓分類、分群、檢索等下游任務更簡單，且能遷移；好表徵的直覺是相似資料自然靠在一起。
- VAE 的損失是重建 + KL；β-VAE 在 KL 前加權重 β，β 變大更解耦但重建變糊，是「解耦 vs 還原」的拉鋸。
- 評估解耦最直接的工具是潛在遍歷：固定其他維度、只掃動一個維度，看是否「一個旋鈕只改一件事」。
- 擴散模型把生成拆成前向加噪與反向去噪的漸進步驟，與 VAE 的一次往返形成對照。
- 生成模型沒有萬用解；用品質、多樣性、可控性、穩定度四面向比較 VAE / GAN / 自回歸 / 擴散，建立選型直覺。
- 實務是一條工作流：表徵 + 生成 + 條件控制 + 評估；條件生成與潛在操控決定輸出，人工把關負責最終篩選與排序。